# EXP-0003: Walk-Forward Probability Calibration Benchmark

**Target:** Post-hoc probability calibration of static pre-race winner models (Model 3 Logistic Regression and Model 4 LightGBM GBDT) under chronological walk-forward conditions.

### Protocol
- **Input Data:** Candidate predictions from EXP-0002 (346 rounds, 2010–2026, 7,285 candidate rows).
- **Calibrators:**
  1. Platt Scaling: Logistic fit on candidate logit-probabilities with within-race normalization.
  2. Isotonic Regression: Monotonic step function with $10^{-6}$ floor and within-race normalization.
  3. Temperature Scaling: Historical race-level multiclass NLL minimization: $p_i(T) = \text{softmax}(\log(p_i) / T)$.
- **Temporal Enforcement:**
  - Expanding chronological window.
  - Initial 20 uniform cold-start races excluded from training data.
  - Minimum 30 valid prior trained races required before calibrating; otherwise raw probabilities retained.
- **Evaluation Split:**
  - Primary Cohort: 2018–2024 (149 rounds).
  - Full Series: 2010–2026 (346 rounds).
- **Inference:** Paired race-clustered bootstrap (2,000 resamples, random seed 42) for 95% confidence intervals on $\Delta \text{LogLoss}$ and $\Delta \text{Brier}$.


In [1]:
# 1. Environment Setup & Project Path Discovery
import os
import sys
import json
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import logsumexp, softmax
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "AGENTS.md").is_file() and (p / "README.md").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate APEX project root containing AGENTS.md and README.md.")

EXP0002_DIR = PROJECT_ROOT / "artifacts/reports/EXP-0002"
M3_CSV_PATH = EXP0002_DIR / "model3_driver_constructor_predictions.csv"
M4_CSV_PATH = EXP0002_DIR / "model4_lightgbm_predictions.csv"

assert M3_CSV_PATH.is_file(), f"Missing Model 3 predictions at {M3_CSV_PATH}"
assert M4_CSV_PATH.is_file(), f"Missing Model 4 predictions at {M4_CSV_PATH}"

EXP0003_DIR = PROJECT_ROOT / "artifacts/reports/EXP-0003"
EXP0003_DIR.mkdir(parents=True, exist_ok=True)

BIN_EDGES = np.array([
    0.00, 0.02, 0.04, 0.06, 0.08, 0.10,
    0.15, 0.20, 0.30, 0.50, 1.000000001
])

MIN_CALIBRATION_RACES = 30
RANDOM_SEED = 42

print(f"APEX Root: {PROJECT_ROOT}")
print(f"Artifacts Destination: {EXP0003_DIR.relative_to(PROJECT_ROOT)}")


APEX Root: /Users/dhruv/dhruv files/apex-f1
Artifacts Destination: artifacts/reports/EXP-0003


## 2. Input Loading & Verification

Verifies schema consistency, 1-to-1 entrant alignment, candidate uniqueness, winner counts, and probability simplex properties between Model 3 and Model 4 outputs.


In [2]:
# 2. Load Model Predictions & Verify Schemas
df_m3 = pd.read_csv(M3_CSV_PATH)
df_m4 = pd.read_csv(M4_CSV_PATH)

assert len(df_m3) == len(df_m4) == 7285, f"Unexpected row count: {len(df_m3)}"
assert (df_m3["race_id"] == df_m4["race_id"]).all(), "Race ID mismatch"
assert (df_m3["driver_id"] == df_m4["driver_id"]).all(), "Driver ID mismatch"
assert (df_m3["y"] == df_m4["y"]).all(), "Winner label mismatch"
assert not df_m3.duplicated(["race_id", "driver_id"]).any(), "Duplicate entrants in Model 3"
assert not df_m4.duplicated(["race_id", "driver_id"]).any(), "Duplicate entrants in Model 4"

winners = df_m3.groupby("race_id")["y"].sum()
assert winners.eq(1).all(), "Every race must have exactly one recorded winner"

for name, df in [("Model 3", df_m3), ("Model 4", df_m4)]:
    sums = df.groupby("race_id")["probability"].sum()
    assert float((sums - 1.0).abs().max()) < 1e-4, f"Simplex violation in {name}"
    assert df["probability"].between(0.0, 1.0).all(), f"Probabilities out of bounds in {name}"

print("Input predictions verified across 346 races and 7,285 candidate records.")


Input predictions verified across 346 races and 7,285 candidate records.


## 3. Walk-Forward Calibration Implementation

Implements expanding walk-forward calibration for:
- Platt Scaling: $\tilde{p}_i = \sigma(a z_i + b)$ where $z_i = \text{logit}(p_i)$.
- Isotonic Regression: Monotonic step function with floor $\max(\tilde{p}_i, 10^{-6})$.
- Temperature Scaling: $\text{NLL}(T) = -\sum_{r=1}^R \log p_{r, w_r}(T)$, $p_i^*(T) = \text{softmax}(\log(p_i) / T)$.

Initial 20 cold-start races are excluded from training history; minimum 30 valid prior races required before calibrating.


In [3]:
# 3. Walk-Forward Calibration Engine Implementation
def run_chronological_calibration(df_input, model_name, min_history=MIN_CALIBRATION_RACES):
    race_schedule = (
        df_input[["race_id", "season_int", "round_int", "race_date", "prediction_method"]]
        .drop_duplicates()
        .sort_values(["season_int", "round_int"])
        .reset_index(drop=True)
    )
    
    race_groups = {rid: grp.copy() for rid, grp in df_input.groupby("race_id", sort=False)}
    
    calibrated_rows = []
    audit_records = []
    valid_prior_races = []
    eps_logit = 1e-12
    
    for idx, rrow in race_schedule.iterrows():
        rid = rrow["race_id"]
        grp = race_groups[rid].copy()
        raw_p = grp["probability"].to_numpy(dtype=float)
        raw_y = grp["y"].to_numpy(dtype=int)
        
        if len(valid_prior_races) < min_history:
            grp["prob_raw"] = raw_p
            grp["prob_platt"] = raw_p
            grp["prob_isotonic"] = raw_p
            grp["prob_temperature"] = raw_p
            grp["cal_status"] = "uncalibrated_warmup"
            grp["n_valid_prior_races"] = len(valid_prior_races)
            grp["temperature"] = 1.0
            calibrated_rows.append(grp)
            
            audit_records.append({
                "race_id": rid,
                "season": int(rrow["season_int"]),
                "round": int(rrow["round_int"]),
                "race_date": str(rrow["race_date"]),
                "model_name": model_name,
                "n_valid_prior_races": len(valid_prior_races),
                "cal_status": "uncalibrated_warmup",
                "platt_coef": np.nan,
                "platt_intercept": np.nan,
                "isotonic_fitted": False,
                "temperature": 1.0,
                "raw_winner_prob": float(raw_p[raw_y == 1][0]),
                "platt_winner_prob": float(raw_p[raw_y == 1][0]),
                "isotonic_winner_prob": float(raw_p[raw_y == 1][0]),
                "temp_winner_prob": float(raw_p[raw_y == 1][0]),
            })
        else:
            hist_dfs = [race_groups[hrid] for hrid in valid_prior_races]
            hist_df = pd.concat(hist_dfs, ignore_index=True)
            
            hist_p = hist_df["probability"].to_numpy(dtype=float)
            hist_y = hist_df["y"].to_numpy(dtype=int)
            
            # Platt Scaling
            h_p_clip = np.clip(hist_p, eps_logit, 1.0 - eps_logit)
            h_z = np.log(h_p_clip / (1.0 - h_p_clip))
            r_p_clip = np.clip(raw_p, eps_logit, 1.0 - eps_logit)
            r_z = np.log(r_p_clip / (1.0 - r_p_clip))
            
            lr = LogisticRegression(C=1.0, solver="lbfgs", random_state=RANDOM_SEED)
            lr.fit(h_z[:, None], hist_y)
            platt_scores = lr.predict_proba(r_z[:, None])[:, 1]
            p_platt = platt_scores / np.sum(platt_scores)
            
            # Isotonic Regression
            iso = IsotonicRegression(y_min=0.0, y_max=1.0, increasing=True, out_of_bounds="clip")
            iso.fit(hist_p, hist_y)
            iso_scores = iso.predict(raw_p)
            if np.sum(iso_scores) <= 1e-8:
                p_iso = raw_p.copy()
            else:
                iso_scores_safe = np.maximum(iso_scores, 1e-6)
                p_iso = iso_scores_safe / np.sum(iso_scores_safe)
                
            # Temperature Scaling
            hist_race_logits = []
            for hrid in valid_prior_races:
                hgrp = race_groups[hrid]
                hp = hgrp["probability"].to_numpy(dtype=float)
                hy = hgrp["y"].to_numpy(dtype=int)
                w_pos = np.where(hy == 1)[0][0]
                log_hp = np.log(np.clip(hp, 1e-15, 1.0))
                hist_race_logits.append((log_hp, w_pos))
                
            def nll_obj(T):
                nll = 0.0
                inv_T = 1.0 / T
                for log_hp, w_pos in hist_race_logits:
                    scaled = log_hp * inv_T
                    lse = logsumexp(scaled)
                    nll -= (scaled[w_pos] - lse)
                return nll
                
            res_opt = minimize_scalar(nll_obj, bounds=(0.05, 10.0), method="bounded")
            opt_T = float(res_opt.x)
            
            raw_log_p = np.log(np.clip(raw_p, 1e-15, 1.0))
            scaled_curr = raw_log_p / opt_T
            p_temp = np.exp(scaled_curr - logsumexp(scaled_curr))
            p_temp = p_temp / np.sum(p_temp)
            
            grp["prob_raw"] = raw_p
            grp["prob_platt"] = p_platt
            grp["prob_isotonic"] = p_iso
            grp["prob_temperature"] = p_temp
            grp["cal_status"] = "calibrated"
            grp["n_valid_prior_races"] = len(valid_prior_races)
            grp["temperature"] = opt_T
            calibrated_rows.append(grp)
            
            audit_records.append({
                "race_id": rid,
                "season": int(rrow["season_int"]),
                "round": int(rrow["round_int"]),
                "race_date": str(rrow["race_date"]),
                "model_name": model_name,
                "n_valid_prior_races": len(valid_prior_races),
                "cal_status": "calibrated",
                "platt_coef": float(lr.coef_[0][0]),
                "platt_intercept": float(lr.intercept_[0]),
                "isotonic_fitted": True,
                "temperature": opt_T,
                "raw_winner_prob": float(raw_p[raw_y == 1][0]),
                "platt_winner_prob": float(p_platt[raw_y == 1][0]),
                "isotonic_winner_prob": float(p_iso[raw_y == 1][0]),
                "temp_winner_prob": float(p_temp[raw_y == 1][0]),
            })
            
        if rrow["prediction_method"] != "uniform_cold_start":
            valid_prior_races.append(rid)
            
    df_calibrated = pd.concat(calibrated_rows, ignore_index=True)
    df_audit = pd.DataFrame(audit_records)
    return df_calibrated, df_audit


## 4. Walk-Forward Execution & Simplex Verification

Executes walk-forward calibration across all 346 races and asserts probability simplex constraints ($\sum p_i = 1$, $0 \le p_i \le 1$).


In [4]:
# 4. Execute Calibration on Model 3 and Model 4
df_m3_cal, df_m3_audit = run_chronological_calibration(df_m3, "Model 3 (Logistic)")
df_m4_cal, df_m4_audit = run_chronological_calibration(df_m4, "Model 4 (LightGBM)")

prob_cols = ["prob_raw", "prob_platt", "prob_isotonic", "prob_temperature"]
for name, df_c in [("Model 3", df_m3_cal), ("Model 4", df_m4_cal)]:
    for col in prob_cols:
        assert df_c[col].notna().all(), f"NaN in {name} {col}"
        assert df_c[col].between(0.0, 1.0).all(), f"Out of bounds in {name} {col}"
        sums = df_c.groupby("race_id")[col].sum()
        assert float((sums - 1.0).abs().max()) < 1e-4, f"Simplex error in {name} {col}"

print("All simplex checks passed.")
m3_cal_mean = float(df_m3_cal.loc[df_m3_cal['cal_status'] == 'calibrated', 'temperature'].mean())
m4_cal_mean = float(df_m4_cal.loc[df_m4_cal['cal_status'] == 'calibrated', 'temperature'].mean())
print(f"Model 3 Temperature (calibrated mean): {m3_cal_mean:.4f}")
print(f"Model 4 Temperature (calibrated mean): {m4_cal_mean:.4f}")


All simplex checks passed.
Model 3 Temperature (calibrated mean): 1.0071
Model 4 Temperature (calibrated mean): 1.3597


## 5. Proper Scoring & Calibration Evaluation

Evaluates multiclass log loss, multiclass Brier score, fractional top-1 accuracy, tie-aware top-1 accuracy, and fixed-bin calibration metrics across Primary Cohort (2018–2024, 149 rounds) and Full Series (2010–2026, 346 rounds).


In [5]:
# 5. Evaluation Metric Computations
def evaluate_multiclass_metrics(df_cand, prob_col):
    log_losses = []
    brier_scores = []
    tie_aware_top1 = 0
    fractional_top1 = 0.0
    total_races = 0

    for race_id, group in df_cand.groupby("race_id", sort=False):
        p = group[prob_col].to_numpy(dtype=float)
        y = group["y"].to_numpy(dtype=int)
        
        winner_idx = np.where(y == 1)[0]
        assert len(winner_idx) == 1
        w_pos = winner_idx[0]
        
        p_winner = np.clip(p[w_pos], 1e-15, 1.0)
        log_losses.append(-np.log(p_winner))
        brier_scores.append(np.sum((p - y) ** 2))
        
        max_p = np.max(p)
        top_indices = np.where(np.isclose(p, max_p, atol=1e-6))[0]
        if w_pos in top_indices:
            tie_aware_top1 += 1
            fractional_top1 += 1.0 / len(top_indices)
        total_races += 1

    return {
        "races": total_races,
        "mean_log_loss": float(np.mean(log_losses)),
        "mean_brier_score": float(np.mean(brier_scores)),
        "tie_aware_top1": float(tie_aware_top1 / total_races),
        "fractional_top1": float(fractional_top1 / total_races),
        "race_log_losses": np.array(log_losses),
        "race_briers": np.array(brier_scores),
    }

def compute_calibration_table(p_arr, y_arr, edges=BIN_EDGES):
    n_total = len(p_arr)
    bin_id = np.digitize(p_arr, edges[1:-1], right=False)
    rows = []
    for i in range(len(edges) - 1):
        mask = (bin_id == i)
        count = int(mask.sum())
        b_lower = float(edges[i])
        b_upper = float(min(edges[i + 1], 1.0))
        label = f"[{b_lower:.2f}, {b_upper:.2f})"
        if count > 0:
            mean_pred = float(p_arr[mask].mean())
            obs_freq = float(y_arr[mask].mean())
            abs_gap = float(abs(obs_freq - mean_pred))
            weight = float(count / n_total)
            rows.append({
                "bin_index": i, "bin_lower": b_lower, "bin_upper": b_upper, "bin_label": label,
                "n_candidates": count, "weight": weight, "mean_predicted_probability": mean_pred,
                "observed_win_frequency": obs_freq, "calibration_gap": float(obs_freq - mean_pred),
                "absolute_gap": abs_gap, "is_empty": False,
            })
        else:
            rows.append({
                "bin_index": i, "bin_lower": b_lower, "bin_upper": b_upper, "bin_label": label,
                "n_candidates": 0, "weight": 0.0, "mean_predicted_probability": np.nan,
                "observed_win_frequency": np.nan, "calibration_gap": np.nan,
                "absolute_gap": np.nan, "is_empty": True,
            })
    df_all = pd.DataFrame(rows)
    df_non_empty = df_all[~df_all["is_empty"]].copy()
    ece = float((df_non_empty["weight"] * df_non_empty["absolute_gap"]).sum())
    mce = float(df_non_empty["absolute_gap"].max())
    citl = float(y_arr.mean() - p_arr.mean())
    return df_all, df_non_empty, {"fixed_bin_ECE": ece, "fixed_bin_MCE": mce, "CITL": citl}

eval_records = []
for cohort_name, filter_fn in [
    ("Primary Cohort (2018-2024)", lambda df: (df["season_int"] >= 2018) & (df["season_int"] <= 2024)),
    ("Full Series (2010-2026)", lambda df: df["season_int"] >= 2010),
]:
    for m_label, df_cal in [("Model 3 (Logistic)", df_m3_cal), ("Model 4 (LightGBM)", df_m4_cal)]:
        sub_df = df_cal[filter_fn(df_cal)].copy()
        y_vec = sub_df["y"].to_numpy(dtype=int)
        
        for m_variant, col in [
            ("Raw (Uncalibrated)", "prob_raw"),
            ("Platt Scaling", "prob_platt"),
            ("Isotonic Regression", "prob_isotonic"),
            ("Temperature Scaling", "prob_temperature"),
        ]:
            m_res = evaluate_multiclass_metrics(sub_df, col)
            p_vec = sub_df[col].to_numpy(dtype=float)
            _, _, cal_summary = compute_calibration_table(p_vec, y_vec)
            
            eval_records.append({
                "Cohort": cohort_name,
                "Base Model": m_label,
                "Method": m_variant,
                "Races": m_res["races"],
                "Log Loss": m_res["mean_log_loss"],
                "Brier Score": m_res["mean_brier_score"],
                "Fract Top-1": m_res["fractional_top1"],
                "Tie-Aware Top-1": m_res["tie_aware_top1"],
                "ECE": cal_summary["fixed_bin_ECE"],
                "MCE": cal_summary["fixed_bin_MCE"],
                "CITL": cal_summary["CITL"],
            })

df_eval_summary = pd.DataFrame(eval_records)
print(df_eval_summary.to_string(index=False))


                    Cohort         Base Model              Method  Races  Log Loss  Brier Score  Fract Top-1  Tie-Aware Top-1      ECE      MCE         CITL
Primary Cohort (2018-2024) Model 3 (Logistic)  Raw (Uncalibrated)    149  1.355201     0.612698     0.516779         0.516779 0.009618 0.073911 3.469447e-17
Primary Cohort (2018-2024) Model 3 (Logistic)       Platt Scaling    149  1.357407     0.613388     0.516779         0.516779 0.009385 0.078753 0.000000e+00
Primary Cohort (2018-2024) Model 3 (Logistic) Isotonic Regression    149  1.403784     0.613623     0.533557         0.583893 0.005006 0.117718 0.000000e+00
Primary Cohort (2018-2024) Model 3 (Logistic) Temperature Scaling    149  1.357764     0.613515     0.516779         0.516779 0.009560 0.077360 0.000000e+00
Primary Cohort (2018-2024) Model 4 (LightGBM)  Raw (Uncalibrated)    149  1.409149     0.628561     0.530201         0.530201 0.013875 0.133525 3.469447e-17
Primary Cohort (2018-2024) Model 4 (LightGBM)       Platt 

## 6. Paired Race-Clustered Bootstrap Inference

Executes paired race-clustered bootstrapping (2,000 resamples, random seed 42) across the 149 primary cohort races to evaluate confidence intervals for differences in Log Loss and Brier Score.


In [6]:
# 6. Paired Race-Clustered Bootstrap Engine (2,000 Replicates, Seed 42)
rng = np.random.default_rng(RANDOM_SEED)

primary_m3 = df_m3_cal[(df_m3_cal["season_int"] >= 2018) & (df_m3_cal["season_int"] <= 2024)].copy()
primary_m4 = df_m4_cal[(df_m4_cal["season_int"] >= 2018) & (df_m4_cal["season_int"] <= 2024)].copy()

race_ids_primary = primary_m3["race_id"].drop_duplicates().values
n_primary_races = len(race_ids_primary)
assert n_primary_races == 149, f"Expected 149 races, got {n_primary_races}"

race_perf = {}
for name, sub_df in [("m3", primary_m3), ("m4", primary_m4)]:
    race_perf[name] = {}
    for col in prob_cols:
        ll_list = []
        bs_list = []
        for rid in race_ids_primary:
            grp = sub_df[sub_df["race_id"] == rid]
            p = grp[col].to_numpy(dtype=float)
            y = grp["y"].to_numpy(dtype=int)
            w_pos = np.where(y == 1)[0][0]
            ll_list.append(-np.log(np.clip(p[w_pos], 1e-15, 1.0)))
            bs_list.append(np.sum((p - y) ** 2))
        race_perf[name][col] = {
            "log_loss": np.array(ll_list),
            "brier": np.array(bs_list),
        }

boot_cluster_idx = rng.choice(n_primary_races, size=(2000, n_primary_races), replace=True)

test_comparisons = [
    ("Model 3: Platt vs Raw", race_perf["m3"]["prob_platt"], race_perf["m3"]["prob_raw"]),
    ("Model 3: Isotonic vs Raw", race_perf["m3"]["prob_isotonic"], race_perf["m3"]["prob_raw"]),
    ("Model 3: Temperature vs Raw", race_perf["m3"]["prob_temperature"], race_perf["m3"]["prob_raw"]),
    ("Model 4: Platt vs Raw", race_perf["m4"]["prob_platt"], race_perf["m4"]["prob_raw"]),
    ("Model 4: Isotonic vs Raw", race_perf["m4"]["prob_isotonic"], race_perf["m4"]["prob_raw"]),
    ("Model 4: Temperature vs Raw", race_perf["m4"]["prob_temperature"], race_perf["m4"]["prob_raw"]),
    ("Calibrated Comparison: M4 Temp vs M3 Temp", race_perf["m4"]["prob_temperature"], race_perf["m3"]["prob_temperature"]),
    ("Calibrated Comparison: M4 Platt vs M3 Platt", race_perf["m4"]["prob_platt"], race_perf["m3"]["prob_platt"]),
]

bootstrap_results = []
for comp_name, m_test, m_base in test_comparisons:
    diff_ll_boot = np.mean(m_test["log_loss"][boot_cluster_idx], axis=1) - np.mean(m_base["log_loss"][boot_cluster_idx], axis=1)
    diff_bs_boot = np.mean(m_test["brier"][boot_cluster_idx], axis=1) - np.mean(m_base["brier"][boot_cluster_idx], axis=1)
    
    point_ll = float(np.mean(m_test["log_loss"]) - np.mean(m_base["log_loss"]))
    point_bs = float(np.mean(m_test["brier"]) - np.mean(m_base["brier"]))
    
    bootstrap_results.append({
        "Comparison": comp_name,
        "Delta_LogLoss": point_ll,
        "LL_95_CI": f"[{np.percentile(diff_ll_boot, 2.5):.4f}, {np.percentile(diff_ll_boot, 97.5):.4f}]",
        "LL_Sig": bool((np.percentile(diff_ll_boot, 2.5) > 0) or (np.percentile(diff_ll_boot, 97.5) < 0)),
        "Delta_Brier": point_bs,
        "Brier_95_CI": f"[{np.percentile(diff_bs_boot, 2.5):.4f}, {np.percentile(diff_bs_boot, 97.5):.4f}]",
        "Brier_Sig": bool((np.percentile(diff_bs_boot, 2.5) > 0) or (np.percentile(diff_bs_boot, 97.5) < 0)),
    })

df_boot_summary = pd.DataFrame(bootstrap_results)
print(df_boot_summary.to_string(index=False))


                                 Comparison  Delta_LogLoss          LL_95_CI  LL_Sig  Delta_Brier       Brier_95_CI  Brier_Sig
                      Model 3: Platt vs Raw       0.002206 [-0.0005, 0.0048]   False     0.000690 [-0.0003, 0.0017]      False
                   Model 3: Isotonic vs Raw       0.048584 [-0.0217, 0.1690]   False     0.000925 [-0.0084, 0.0109]      False
                Model 3: Temperature vs Raw       0.002563  [0.0003, 0.0048]    True     0.000818 [-0.0001, 0.0018]      False
                      Model 4: Platt vs Raw      -0.020486 [-0.0808, 0.0340]   False    -0.006037 [-0.0245, 0.0118]      False
                   Model 4: Isotonic vs Raw       0.040891 [-0.0378, 0.1407]   False    -0.005353 [-0.0248, 0.0127]      False
                Model 4: Temperature vs Raw      -0.021562 [-0.0818, 0.0331]   False    -0.005276 [-0.0257, 0.0142]      False
  Calibrated Comparison: M4 Temp vs M3 Temp       0.029823 [-0.0584, 0.1208]   False     0.009770 [-0.0218, 0.0

## 7. Regulation Era Stratification

Stratifies calibration metrics across technical eras:
- V8 Era (2010–2013)
- Turbo-Hybrid Era (2014–2021)
- Ground-Effect Era (2022–2026)


In [7]:
# 7. Stratified Analysis Across Regulation Eras
def get_era(season):
    if season <= 2013:
        return "V8 Era (2010-2013)"
    elif season <= 2021:
        return "Turbo-Hybrid (2014-2021)"
    else:
        return "Ground-Effect (2022-2026)"

df_m3_cal["era"] = df_m3_cal["season_int"].apply(get_era)
df_m4_cal["era"] = df_m4_cal["season_int"].apply(get_era)

era_records = []
for era_name in ["V8 Era (2010-2013)", "Turbo-Hybrid (2014-2021)", "Ground-Effect (2022-2026)"]:
    for m_label, df_cal in [("Model 3", df_m3_cal), ("Model 4", df_m4_cal)]:
        era_df = df_cal[df_cal["era"] == era_name]
        if len(era_df) == 0:
            continue
        y_v = era_df["y"].to_numpy(dtype=int)
        for col_name, ckey in [("Raw", "prob_raw"), ("Temperature", "prob_temperature")]:
            m_res = evaluate_multiclass_metrics(era_df, ckey)
            _, _, c_sum = compute_calibration_table(era_df[ckey].to_numpy(dtype=float), y_v)
            era_records.append({
                "Era": era_name,
                "Model": m_label,
                "Variant": col_name,
                "Races": m_res["races"],
                "Log Loss": m_res["mean_log_loss"],
                "Brier Score": m_res["mean_brier_score"],
                "ECE": c_sum["fixed_bin_ECE"],
                "MCE": c_sum["fixed_bin_MCE"],
            })

df_era_summary = pd.DataFrame(era_records)
print(df_era_summary.to_string(index=False))


                      Era   Model     Variant  Races  Log Loss  Brier Score      ECE      MCE
       V8 Era (2010-2013) Model 3         Raw     77  1.839411     0.691448 0.005864 0.110022
       V8 Era (2010-2013) Model 3 Temperature     77  1.845630     0.693661 0.006148 0.113683
       V8 Era (2010-2013) Model 4         Raw     77  2.057034     0.767864 0.012135 0.141099
       V8 Era (2010-2013) Model 4 Temperature     77  2.056746     0.773377 0.012087 0.150906
 Turbo-Hybrid (2014-2021) Model 3         Raw    160  1.333930     0.627330 0.005854 0.082381
 Turbo-Hybrid (2014-2021) Model 3 Temperature    160  1.337634     0.628154 0.005616 0.077139
 Turbo-Hybrid (2014-2021) Model 4         Raw    160  1.546161     0.682941 0.016112 0.174425
 Turbo-Hybrid (2014-2021) Model 4 Temperature    160  1.477736     0.666534 0.006740 0.051166
Ground-Effect (2022-2026) Model 3         Raw    109  1.248101     0.573317 0.008940 0.121767
Ground-Effect (2022-2026) Model 3 Temperature    109  1.2472

## 8. Diagnostic Visualizations

Generates reliability curves and temperature parameter traces saved to `artifacts/reports/EXP-0003/`.


In [8]:
# 8. Plot Reliability Diagrams and Parameter Traces
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

def plot_model_calibration(df_cal, model_title, output_filename):
    sub = df_cal[(df_cal["season_int"] >= 2018) & (df_cal["season_int"] <= 2024)].copy()
    y_true = sub["y"].to_numpy(dtype=int)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    ax1 = axes[0]
    ax1.plot([0, 1], [0, 1], "k--", label="Perfect Calibration", alpha=0.7)
    
    colors = {"prob_raw": "red", "prob_platt": "blue", "prob_isotonic": "green", "prob_temperature": "purple"}
    labels = {"prob_raw": "Raw", "prob_platt": "Platt", "prob_isotonic": "Isotonic", "prob_temperature": "Temperature"}
    
    for c_col, color in colors.items():
        _, df_ne, s_dict = compute_calibration_table(sub[c_col].to_numpy(dtype=float), y_true)
        ax1.plot(
            df_ne["mean_predicted_probability"],
            df_ne["observed_win_frequency"],
            marker="o",
            color=color,
            label=f"{labels[c_col]} (ECE={s_dict['fixed_bin_ECE']:.4f})"
        )
        
    ax1.set_xlim([-0.02, 0.85])
    ax1.set_ylim([-0.02, 0.85])
    ax1.set_xlabel("Mean Predicted Probability", fontsize=11)
    ax1.set_ylabel("Observed Win Frequency", fontsize=11)
    ax1.set_title(f"Reliability Diagram: {model_title} (2018-2024)", fontsize=12)
    ax1.legend(loc="upper left")
    
    ax2 = axes[1]
    for c_col, color in [("prob_raw", "red"), ("prob_temperature", "purple")]:
        ax2.hist(
            sub[c_col],
            bins=np.linspace(0, 0.8, 40),
            alpha=0.4,
            color=color,
            label=labels[c_col],
            density=True
        )
    ax2.set_xlabel("Predicted Probability", fontsize=11)
    ax2.set_ylabel("Density", fontsize=11)
    ax2.set_title("Probability Distribution (Raw vs Temperature)", fontsize=12)
    ax2.legend()
    
    plt.tight_layout()
    fig.savefig(EXP0003_DIR / output_filename, dpi=200)
    plt.close(fig)

plot_model_calibration(df_m3_cal, "Model 3 (Logistic)", "reliability_diagrams_model3.png")
plot_model_calibration(df_m4_cal, "Model 4 (LightGBM)", "reliability_diagrams_model4.png")

fig, ax = plt.subplots(figsize=(10, 5))
sub_audit_m3 = df_m3_audit[df_m3_audit["cal_status"] == "calibrated"]
sub_audit_m4 = df_m4_audit[df_m4_audit["cal_status"] == "calibrated"]

ax.plot(pd.to_datetime(sub_audit_m3["race_date"]), sub_audit_m3["temperature"], label="Model 3 Temperature (Logistic)", color="blue")
ax.plot(pd.to_datetime(sub_audit_m4["race_date"]), sub_audit_m4["temperature"], label="Model 4 Temperature (LightGBM)", color="orange")
ax.axhline(1.0, color="black", linestyle="--", alpha=0.7, label="Neutral Temperature (T=1.0)")
ax.set_xlabel("Grand Prix Date", fontsize=11)
ax.set_ylabel("Optimal Temperature T", fontsize=11)
ax.set_title("Chronological Temperature Evolution", fontsize=12)
ax.legend()
plt.tight_layout()
fig.savefig(EXP0003_DIR / "temperature_trajectories.png", dpi=200)
plt.close(fig)

print("Diagnostic figures generated.")


Diagnostic figures generated.


## 9. Serialization

Serializes prediction datasets, audit records, evaluation metrics, and bootstrap results under `artifacts/reports/EXP-0003/`.


In [9]:
# 9. Serialization of Artifacts
df_m3_cal.to_csv(EXP0003_DIR / "model3_calibrated_predictions.csv", index=False)
df_m4_cal.to_csv(EXP0003_DIR / "model4_calibrated_predictions.csv", index=False)

audit_combined = pd.concat([df_m3_audit, df_m4_audit], ignore_index=True)
audit_combined.to_csv(EXP0003_DIR / "calibration_walk_forward_audit.csv", index=False)

binned_rows = []
for m_label, df_cal in [("Model 3", df_m3_cal), ("Model 4", df_m4_cal)]:
    sub = df_cal[(df_cal["season_int"] >= 2018) & (df_cal["season_int"] <= 2024)]
    y_v = sub["y"].to_numpy(dtype=int)
    for v_name, col in [("Raw", "prob_raw"), ("Platt", "prob_platt"), ("Isotonic", "prob_isotonic"), ("Temperature", "prob_temperature")]:
        df_all_b, _, _ = compute_calibration_table(sub[col].to_numpy(dtype=float), y_v)
        df_all_b["model"] = m_label
        df_all_b["variant"] = v_name
        binned_rows.append(df_all_b)
df_all_bins = pd.concat(binned_rows, ignore_index=True)
df_all_bins.to_csv(EXP0003_DIR / "reliability_bins.csv", index=False)

with open(EXP0003_DIR / "bootstrap_results.json", "w") as f:
    json.dump(bootstrap_results, f, indent=2)

with open(EXP0003_DIR / "calibration_comparison.json", "w") as f:
    json.dump(df_eval_summary.to_dict(orient="records"), f, indent=2)

manifest = {
    "experiment_id": "EXP-0003",
    "experiment_title": "Walk-Forward Probability Calibration Benchmark",
    "base_models_evaluated": ["Model 3 (Logistic)", "Model 4 (LightGBM)"],
    "calibration_methods": ["Platt Scaling", "Isotonic Regression", "Temperature Scaling"],
    "primary_cohort": "2018-2024 (149 rounds)",
    "full_series": "2010-2026 (346 rounds)",
    "artifacts_generated": [
        "model3_calibrated_predictions.csv",
        "model4_calibrated_predictions.csv",
        "calibration_walk_forward_audit.csv",
        "reliability_bins.csv",
        "bootstrap_results.json",
        "calibration_comparison.json",
        "reliability_diagrams_model3.png",
        "reliability_diagrams_model4.png",
        "temperature_trajectories.png",
        "manifest.json"
    ]
}
with open(EXP0003_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)

print("Artifacts serialized.")


Artifacts serialized.


## 10. Empirical Findings

### Primary Cohort Summary (2018–2024, 149 Races)
1. **Model 4 (LightGBM) Point Improvement under Temperature & Platt Scaling:**
   - Raw Model 4 Log Loss decreases from 1.4091 to 1.3876 under Temperature Scaling ($T \approx 1.360$) and 1.3887 under Platt Scaling.
   - Brier score decreases from 0.6286 to 0.6225 under Platt Scaling and 0.6233 under Temperature Scaling.
   - Fixed-bin ECE decreases from 0.0139 to 0.0073 (Platt) and 0.0078 (Temperature).
   - Fractional Top-1 accuracy is preserved at 53.02% (79 / 149).
   - In paired race-clustered bootstrap tests (2,000 resamples), the 95% confidence interval for Log Loss change spans zero ($\Delta = -0.0216$, 95% CI $[-0.0818, +0.0331]$); inter-race outcome variance exceeds the magnitude of the point improvement.
2. **Model 3 (Logistic Regression) Invariance:**
   - Raw Model 3 is already close to canonical temperature ($T \approx 1.007$).
   - Calibration changes are minor ($\Delta \text{LogLoss} < +0.0026$).
   - Model 3 Raw Log Loss (1.3552) remains lower than calibrated Model 4 (1.3876).
3. **Isotonic Regression Degradation:**
   - Monotonic step functions introduce plateau discontinuities that penalize upset winners, increasing Log Loss to 1.4500 on Model 4 and 1.4038 on Model 3. Non-parametric step calibration without probability floors is ill-suited to multiclass log loss.
